In [1]:
# import os
# import glob

# possible_paths = [
#     "/kaggle/input/biohub-cell-tracking-during-development",
#     "/kaggle/input/competitions/biohub-cell-tracking-during-development"
# ]

# base_dataset_path = next((path for path in possible_paths if os.path.exists(path)), None)

# if not base_dataset_path:
#     print("Error: Dataset root folder not found in known locations.")
# else:
#     train_path = os.path.join(base_dataset_path, "train")
#     if os.path.exists(train_path):
#         zarr_files = glob.glob(os.path.join(train_path, "*.zarr"))
#         if zarr_files:
#             print(f"Found {len(zarr_files)} .zarr files. Top 3 results:")
#             for file in zarr_files[:3]:
#                 print(file)
#         else:
#             print("Error: No .zarr files found in train directory.")
#     else:
#         print("Error: 'train' directory not found.")

In [2]:
# import zarr
# import matplotlib.pyplot as plt

# # Dataset Path Configuration
# zarr_path = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train/6bba_cf35214c.zarr"

# # Lazy Loading Initialization
# root = zarr.open(zarr_path, mode='r')
# image_tensor = root['0']

# # Tensor Architecture Profiling 
# print("Zarr Metadata Inspection")
# print(f"Tensor Shape (T, Z, Y, X) : {image_tensor.shape}")
# print(f"Data Type (Dtype)         : {image_tensor.dtype}")
# print(f"Chunking Size             : {image_tensor.chunks}\n")


# # spesific  data extraction (Silicing)
# t_index = 0
# z_index = 32

# print(f"Extracting 2D slide at T={t_index} and Z={z_index} into memory... ")
# slice_2d = image_tensor[t_index, z_index, :, :]

# # Extraction Result Visualization
# plt.figure(figsize=(8, 8))
# plt.imshow(slice_2d, cmap='gray')
# plt.title(f"Zebrafish Cell | T={t_index}, Z={z_index}")
# plt.axis('off')
# plt.show()

In [3]:
# import os
# import glob
# import numpy as np
# import pandas as pd
# import zarr
# import scipy.ndimage as ndimage
# import torch
# from torch.utils.data import Dataset, DataLoader

# # Custom PyTorch Dataset for 3D Zebrafish cell tracking
# class ZebrafishDataset(Dataset):
#     def __init__(self, zarr_path, csv_path, time_frame=0, mask_sigma=2.0):
#         # Open the Zarr directory for lazy loading (read-only mode)
#         self.root = zarr.open(zarr_path, mode='r')
#         # Access the main 3D image tensor array
#         self.image_tensor = self.root['0']
#         # Set the specific time frame to extract
#         self.time_frame = time_frame
#         # Set the spread (sigma) for the Gaussian blur
#         self.mask_sigma = mask_sigma
        
#         # Load the coordinates file using Pandas
#         self.annotations = pd.read_csv(csv_path)
#         # Filter coordinates to match only the selected time frame
#         self.points = self.annotations[self.annotations['t'] == self.time_frame]

#     def __len__(self):
#         # Return dataset size (hardcoded to 1 for diagnostic purposes)
#         return 1 

#     def _normalize_intensity(self, volume):
#         # Convert image data to float32 for stable neural network processing
#         volume = volume.astype(np.float32)
#         # Find the minimum and maximum pixel values in the array
#         min_val = np.min(volume)
#         max_val = np.max(volume)
        
#         # Apply Min-Max scaling to compress values between 0.0 and 1.0
#         if max_val - min_val > 0:
#             volume = (volume - min_val) / (max_val - min_val)
#         return volume

#     def _generate_gaussian_mask(self, shape, points):
#         # Create an empty 3D matrix filled with zeros (representing the dark background)
#         mask = np.zeros(shape, dtype=np.float32)
        
#         # Loop through the target coordinates and mark cell centroids with 1.0
#         for _, row in points.iterrows():
#             z, y, x = int(row['z']), int(row['y']), int(row['x'])
#             # Ensure coordinates are strictly within the matrix boundaries
#             if 0 <= z < shape[0] and 0 <= y < shape[1] and 0 <= x < shape[2]:
#                 mask[z, y, x] = 1.0
                
#         # Apply 3D Gaussian filter to turn single points into spherical probability blobs
#         mask = ndimage.gaussian_filter(mask, sigma=self.mask_sigma)
#         # Find the peak probability value after blurring
#         max_prob = np.max(mask)
        
#         # Normalize the blurred mask so the peak is exactly 1.0
#         if max_prob > 0:
#             mask = mask / max_prob
#         return mask

#     def __getitem__(self, idx):
#         # Extract the raw 3D volume (Depth, Height, Width) for the target time frame
#         raw_volume = self.image_tensor[self.time_frame, :, :, :]
#         # Normalize the extracted raw volume
#         x_normalized = self._normalize_intensity(raw_volume)
        
#         # Generate the corresponding 3D Ground Truth target mask
#         y_target = self._generate_gaussian_mask(x_normalized.shape, self.points)
        
#         # Convert numpy arrays to PyTorch tensors and inject a channel dimension (C=1)
#         x_tensor = torch.tensor(x_normalized).unsqueeze(0)
#         y_tensor = torch.tensor(y_target).unsqueeze(0)
        
#         return x_tensor, y_tensor


# # Execution Module for Directory Inspection
# if __name__ == "__main__":
#     # Define the target sample ID and root directory
#     sample_id = "6bba_cf35214c"
#     base_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
    
#     # Construct the absolute paths for Zarr and GEFF directories
#     zarr_dir = os.path.join(base_dir, f"{sample_id}.zarr")
#     geff_dir = os.path.join(base_dir, f"{sample_id}.geff")
    
#     # Verify if the GEFF path is a valid directory
#     if os.path.isdir(geff_dir):
#         print(f"Directory Target Acquired: {geff_dir}")
#         print("Scanning contents inside the directory...")
        
#         # List all hidden files inside the GEFF directory
#         contents = os.listdir(geff_dir)
#         for item in contents:
#             print(f"-> {item}")
            
#         print("\nPlease note the target file name printed above so we can link it to Pandas.")
#     else:
#         # Print an error if the directory does not exist
#         print(f"Error: Directory not found at {geff_dir}")

In [4]:
# import os
# import glob

# # Define the absolute path to the training directory
# train_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"

# print("Investigating Dataset Composition...")

# if not os.path.exists(train_dir):
#     print("Error: Train directory not found.")
# else:
#     # Scan all files/folders directly inside the train directory
#     all_items = glob.glob(os.path.join(train_dir, "*"))
    
#     # Categorize items by their file extension
#     extensions_map = {}
#     for item in all_items:
#         # Extract extension (e.g., .zarr, .json, .txt)
#         ext = os.path.splitext(item)[1].lower()
#         if ext == '':
#             ext = 'Directory / No Extension'
            
#         extensions_map[ext] = extensions_map.get(ext, 0) + 1
        
#     print(f"Total items in 'train' folder: {len(all_items)}\n")
#     print("Breakdown by file type:")
#     for ext, count in extensions_map.items():
#         print(f"-> {count} item(s) of type: {ext}")

In [5]:
# import os
# import zarr

# # Diagnostic Module for GEFF Hierarchy
# if __name__ == "__main__":
#     sample_id = "6bba_cf35214c"
#     base_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
#     geff_dir = os.path.join(base_dir, f"{sample_id}.geff")
    
#     # Open the GEFF directory as a hierarchical Zarr Group
#     geff_group = zarr.open(geff_dir, mode='r')
    
#     print("Menganalisis Struktur Hierarki GEFF...")
    
#     # Print the entire tree structure of the Zarr file
#     print(geff_group.tree())

In [6]:
# # Auto-install dependency to survive kernel restarts
# !pip install zarr

# import os
# import numpy as np
# import zarr
# import scipy.ndimage as ndimage
# import torch
# from torch.utils.data import Dataset, DataLoader

# # Custom PyTorch Dataset mapping Zarr image tensors to GEFF Zarr coordinate graphs
# class ZebrafishDataset(Dataset):
#     def __init__(self, zarr_path, geff_path, time_frame=0, mask_sigma=2.0):
#         # Initialize image tensor
#         self.root_img = zarr.open(zarr_path, mode='r')
#         self.image_tensor = self.root_img['0']
        
#         self.time_frame = time_frame
#         self.mask_sigma = mask_sigma
        
#         # Initialize GEFF graph directory
#         self.root_geff = zarr.open(geff_path, mode='r')
        
#         # Extract individual coordinate vectors directly into memory
#         t_vals = self.root_geff['nodes/props/t/values'][:]
#         z_vals = self.root_geff['nodes/props/z/values'][:]
#         y_vals = self.root_geff['nodes/props/y/values'][:]
#         x_vals = self.root_geff['nodes/props/x/values'][:]
        
#         # Create a boolean mask to isolate coordinates matching the exact time frame
#         time_mask = (t_vals == self.time_frame)
        
#         # Filter and store the isolated Z, Y, X coordinates for mask generation
#         self.target_z = z_vals[time_mask]
#         self.target_y = y_vals[time_mask]
#         self.target_x = x_vals[time_mask]

#     def __len__(self):
#         # Return dataset size (diagnostic mode: 1 sample)
#         return 1 

#     def _normalize_intensity(self, volume):
#         # Convert uint16 to float32 and apply Min-Max scaling
#         volume = volume.astype(np.float32)
#         min_val = np.min(volume)
#         max_val = np.max(volume)
        
#         if max_val - min_val > 0:
#             volume = (volume - min_val) / (max_val - min_val)
#         return volume

#     def _generate_gaussian_mask(self, shape):
#         # Initialize an empty 3D probability matrix
#         mask = np.zeros(shape, dtype=np.float32)
        
#         # Zip the filtered vectors and plant 1.0 at every valid centroid coordinate
#         for z, y, x in zip(self.target_z, self.target_y, self.target_x):
#             z, y, x = int(z), int(y), int(x)
#             if 0 <= z < shape[0] and 0 <= y < shape[1] and 0 <= x < shape[2]:
#                 mask[z, y, x] = 1.0
                
#         # Apply a 3D Gaussian filter to expand centroids into spherical targets
#         mask = ndimage.gaussian_filter(mask, sigma=self.mask_sigma)
#         max_prob = np.max(mask)
        
#         if max_prob > 0:
#             mask = mask / max_prob
#         return mask

#     def __getitem__(self, idx):
#         # 1. Extract and normalize raw 3D input volume
#         raw_volume = self.image_tensor[self.time_frame, :, :, :]
#         x_normalized = self._normalize_intensity(raw_volume)
        
#         # 2. Generate target spatial mask
#         y_target = self._generate_gaussian_mask(x_normalized.shape)
        
#         # 3. Convert to PyTorch tensors and expand the Channel dimension
#         x_tensor = torch.tensor(x_normalized).unsqueeze(0)
#         y_tensor = torch.tensor(y_target).unsqueeze(0)
        
#         return x_tensor, y_tensor


# # Execution Module
# if __name__ == "__main__":
#     sample_id = "6bba_cf35214c"
#     base_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
    
#     # Establish absolute paths
#     zarr_file = os.path.join(base_dir, f"{sample_id}.zarr")
#     geff_file = os.path.join(base_dir, f"{sample_id}.geff")
    
#     # Validate dependencies before execution
#     if not os.path.exists(zarr_file) or not os.path.exists(geff_file):
#         print("System Error: Zarr or GEFF directory missing.")
#     else:
#         print("Data architecture validated. Engaging PyTorch DataLoader...")
        
#         # Initialize DataLoader pipeline
#         dataset = ZebrafishDataset(zarr_path=zarr_file, geff_path=geff_file, time_frame=0)
#         dataloader = DataLoader(dataset, batch_size=1, shuffle=False)
        
#         # Print shape and range statistics for the resulting tensors
#         for inputs, targets in dataloader:
#             print(f"Input Tensor  | Shape: {list(inputs.shape)} | Range: {inputs.min():.4f} to {inputs.max():.4f}")
#             print(f"Target Tensor | Shape: {list(targets.shape)} | Range: {targets.min():.4f} to {targets.max():.4f}")
#             break

In [7]:
# import torch
# import torch.nn as nn

# # Reusable block for Convolution -> BatchNorm -> ReLU
# class DoubleConv3D(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.double_conv = nn.Sequential(
#             nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
#             nn.BatchNorm3d(out_channels),
#             nn.ReLU(inplace=True),
#             nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
#             nn.BatchNorm3d(out_channels),
#             nn.ReLU(inplace=True)
#         )

#     def forward(self, x):
#         return self.double_conv(x)

# # Main 3D U-Net Architecture
# class UNet3D(nn.Module):
#     def __init__(self, in_channels=1, out_channels=1, base_features=16):
#         super().__init__()
        
#         # Encoder (Downsampling path)
#         self.enc1 = DoubleConv3D(in_channels, base_features)
#         self.pool1 = nn.MaxPool3d(kernel_size=2, stride=2)
        
#         self.enc2 = DoubleConv3D(base_features, base_features * 2)
#         self.pool2 = nn.MaxPool3d(kernel_size=2, stride=2)
        
#         self.enc3 = DoubleConv3D(base_features * 2, base_features * 4)
#         self.pool3 = nn.MaxPool3d(kernel_size=2, stride=2)
        
#         # Bottleneck (Deepest feature representation)
#         self.bottleneck = DoubleConv3D(base_features * 4, base_features * 8)
        
#         # Decoder (Upsampling path with Skip Connections)
#         self.upconv3 = nn.ConvTranspose3d(base_features * 8, base_features * 4, kernel_size=2, stride=2)
#         self.dec3 = DoubleConv3D(base_features * 8, base_features * 4)
        
#         self.upconv2 = nn.ConvTranspose3d(base_features * 4, base_features * 2, kernel_size=2, stride=2)
#         self.dec2 = DoubleConv3D(base_features * 4, base_features * 2)
        
#         self.upconv1 = nn.ConvTranspose3d(base_features * 2, base_features, kernel_size=2, stride=2)
#         self.dec1 = DoubleConv3D(base_features * 2, base_features)
        
#         # Final output layer to map features back to a single channel probability mask
#         self.out_conv = nn.Conv3d(base_features, out_channels, kernel_size=1)

#     def forward(self, x):
#         # Encoder operations
#         e1 = self.enc1(x)
#         p1 = self.pool1(e1)
        
#         e2 = self.enc2(p1)
#         p2 = self.pool2(e2)
        
#         e3 = self.enc3(p2)
#         p3 = self.pool3(e3)
        
#         # Bottleneck operations
#         b = self.bottleneck(p3)
        
#         # Decoder operations (Concatenating skip connections along the Channel dimension)
#         d3 = self.upconv3(b)
#         d3 = torch.cat((d3, e3), dim=1)
#         d3 = self.dec3(d3)
        
#         d2 = self.upconv2(d3)
#         d2 = torch.cat((d2, e2), dim=1)
#         d2 = self.dec2(d2)
        
#         d1 = self.upconv1(d2)
#         d1 = torch.cat((d1, e1), dim=1)
#         d1 = self.dec1(d1)
        
#         # Map features to the output channel and apply Sigmoid [0.0, 1.0]
#         out = self.out_conv(d1)
#         return torch.sigmoid(out)


# # Execution Module for Architecture Validation
# if __name__ == "__main__":
#     print("Initializing 3D U-Net Architecture...")
    
#     # Instantiate the model
#     model = UNet3D(in_channels=1, out_channels=1, base_features=16)
    
#     # Create a dummy tensor mimicking the exact output from your DataLoader
#     dummy_input = torch.randn(1, 1, 64, 256, 256)
#     print(f"Input Tensor Shape  : {list(dummy_input.shape)}")
    
#     # Execute a forward pass
#     output = model(dummy_input)
#     print(f"Output Tensor Shape : {list(output.shape)}")
    
#     # Verify exact shape match
#     if dummy_input.shape == output.shape:
#         print("Success: Arsitektur 3D U-Net valid dan siap digunakan untuk Training.")
#     else:
#         print("Error: Terjadi distorsi dimensi pada arsitektur.")

In [8]:
# import torch.optim as optim
# import torch.nn as nn

# if __name__ == "__main__":
#     print("Initializing Full Training Engine...")
    
#     # Hardware Allocation
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"Hardware Detected: {device}")
    
#     model = model.to(device)
#     criterion = nn.MSELoss()
#     optimizer = optim.Adam(model.parameters(), lr=1e-4)
    
#     # Define total training cycles (epochs)
#     epochs = 5 
    
#     print("Starting Training Loop...")
    
#     for epoch in range(epochs):
#         model.train()
#         epoch_loss = 0.0
        
#         for batch_idx, (inputs, targets) in enumerate(dataloader):
#             inputs = inputs.to(device)
#             targets = targets.to(device)
            
#             optimizer.zero_grad()
#             predictions = model(inputs)
#             loss = criterion(predictions, targets)
#             loss.backward()
#             optimizer.step()
            
#             # Accumulate loss for monitoring
#             epoch_loss += loss.item()
            
#         # Calculate and print average loss per epoch
#         avg_loss = epoch_loss / len(dataloader)
#         print(f"Epoch [{epoch+1}/{epochs}] | Average Loss: {avg_loss:.6f}")

In [9]:
# # Auto-install dependency 
# !pip install zarr

# import os
# import numpy as np
# import zarr
# import scipy.ndimage as ndimage
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import matplotlib.pyplot as plt
# from torch.utils.data import Dataset, DataLoader

# # 1. Dataset Pipeline
# class ZebrafishDataset(Dataset):
#     def __init__(self, zarr_path, geff_path, time_frame=0, mask_sigma=2.0):
#         self.root_img = zarr.open(zarr_path, mode='r')
#         self.image_tensor = self.root_img['0']
#         self.time_frame = time_frame
#         self.mask_sigma = mask_sigma
        
#         self.root_geff = zarr.open(geff_path, mode='r')
#         t_vals = self.root_geff['nodes/props/t/values'][:]
#         z_vals = self.root_geff['nodes/props/z/values'][:]
#         y_vals = self.root_geff['nodes/props/y/values'][:]
#         x_vals = self.root_geff['nodes/props/x/values'][:]
        
#         time_mask = (t_vals == self.time_frame)
#         self.target_z = z_vals[time_mask]
#         self.target_y = y_vals[time_mask]
#         self.target_x = x_vals[time_mask]

#     def __len__(self):
#         return 1 

#     def _normalize_intensity(self, volume):
#         volume = volume.astype(np.float32)
#         min_val = np.min(volume)
#         max_val = np.max(volume)
#         if max_val - min_val > 0:
#             volume = (volume - min_val) / (max_val - min_val)
#         return volume

#     def _generate_gaussian_mask(self, shape):
#         mask = np.zeros(shape, dtype=np.float32)
#         for z, y, x in zip(self.target_z, self.target_y, self.target_x):
#             z, y, x = int(z), int(y), int(x)
#             if 0 <= z < shape[0] and 0 <= y < shape[1] and 0 <= x < shape[2]:
#                 mask[z, y, x] = 1.0
                
#         mask = ndimage.gaussian_filter(mask, sigma=self.mask_sigma)
#         max_prob = np.max(mask)
#         if max_prob > 0:
#             mask = mask / max_prob
#         return mask

#     def __getitem__(self, idx):
#         raw_volume = self.image_tensor[self.time_frame, :, :, :]
#         x_normalized = self._normalize_intensity(raw_volume)
#         y_target = self._generate_gaussian_mask(x_normalized.shape)
        
#         x_tensor = torch.tensor(x_normalized).unsqueeze(0)
#         y_tensor = torch.tensor(y_target).unsqueeze(0)
#         return x_tensor, y_tensor

# # 2. 3D U-Net Architecture
# class DoubleConv3D(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.double_conv = nn.Sequential(
#             nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
#             nn.BatchNorm3d(out_channels),
#             nn.ReLU(inplace=True),
#             nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
#             nn.BatchNorm3d(out_channels),
#             nn.ReLU(inplace=True)
#         )
#     def forward(self, x):
#         return self.double_conv(x)

# class UNet3D(nn.Module):
#     def __init__(self, in_channels=1, out_channels=1, base_features=16):
#         super().__init__()
#         self.enc1 = DoubleConv3D(in_channels, base_features)
#         self.pool1 = nn.MaxPool3d(kernel_size=2, stride=2)
#         self.enc2 = DoubleConv3D(base_features, base_features * 2)
#         self.pool2 = nn.MaxPool3d(kernel_size=2, stride=2)
#         self.enc3 = DoubleConv3D(base_features * 2, base_features * 4)
#         self.pool3 = nn.MaxPool3d(kernel_size=2, stride=2)
        
#         self.bottleneck = DoubleConv3D(base_features * 4, base_features * 8)
        
#         self.upconv3 = nn.ConvTranspose3d(base_features * 8, base_features * 4, kernel_size=2, stride=2)
#         self.dec3 = DoubleConv3D(base_features * 8, base_features * 4)
#         self.upconv2 = nn.ConvTranspose3d(base_features * 4, base_features * 2, kernel_size=2, stride=2)
#         self.dec2 = DoubleConv3D(base_features * 4, base_features * 2)
#         self.upconv1 = nn.ConvTranspose3d(base_features * 2, base_features, kernel_size=2, stride=2)
#         self.dec1 = DoubleConv3D(base_features * 2, base_features)
        
#         self.out_conv = nn.Conv3d(base_features, out_channels, kernel_size=1)

#     def forward(self, x):
#         e1 = self.enc1(x)
#         p1 = self.pool1(e1)
#         e2 = self.enc2(p1)
#         p2 = self.pool2(e2)
#         e3 = self.enc3(p2)
#         p3 = self.pool3(e3)
        
#         b = self.bottleneck(p3)
        
#         d3 = self.upconv3(b)
#         d3 = torch.cat((d3, e3), dim=1)
#         d3 = self.dec3(d3)
#         d2 = self.upconv2(d3)
#         d2 = torch.cat((d2, e2), dim=1)
#         d2 = self.dec2(d2)
#         d1 = self.upconv1(d2)
#         d1 = torch.cat((d1, e1), dim=1)
#         d1 = self.dec1(d1)
        
#         out = self.out_conv(d1)
#         return torch.sigmoid(out)

# # 3. Master Execution Module
# if __name__ == "__main__":
#     print("Initializing Master Pipeline...")
    
#     # Paths setup
#     sample_id = "6bba_cf35214c"
#     base_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
#     zarr_file = os.path.join(base_dir, f"{sample_id}.zarr")
#     geff_file = os.path.join(base_dir, f"{sample_id}.geff")
    
#     # Initialize hardware, dataloader, and model
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"Hardware Detected: {device}")
    
#     dataset = ZebrafishDataset(zarr_path=zarr_file, geff_path=geff_file, time_frame=0)
#     dataloader = DataLoader(dataset, batch_size=1, shuffle=False)
    
#     model = UNet3D(in_channels=1, out_channels=1, base_features=16).to(device)
#     criterion = nn.MSELoss()
#     optimizer = optim.Adam(model.parameters(), lr=1e-4)
#     epochs = 5 
    
#     # Training Loop
#     print("\nStarting Training Loop...")
#     for epoch in range(epochs):
#         model.train()
#         epoch_loss = 0.0
        
#         for batch_idx, (inputs, targets) in enumerate(dataloader):
#             inputs = inputs.to(device)
#             targets = targets.to(device)
            
#             optimizer.zero_grad()
#             predictions = model(inputs)
#             loss = criterion(predictions, targets)
#             loss.backward()
#             optimizer.step()
            
#             epoch_loss += loss.item()
            
#         avg_loss = epoch_loss / len(dataloader)
#         print(f"Epoch [{epoch+1}/{epochs}] | Average Loss: {avg_loss:.6f}")
        
#     print("\nTraining Complete. Executing Visualization...")
    
#     # Evaluation and Visualization
#     model.eval()
#     with torch.no_grad():
#         for inputs, targets in dataloader:
#             inputs = inputs.to(device)
#             predictions = model(inputs)
            
#             input_vol = inputs.cpu().squeeze().numpy()
#             target_vol = targets.cpu().squeeze().numpy()
#             pred_vol = predictions.cpu().squeeze().numpy()
#             break
            
#     # Plotting results
#     z_slice = 32 
#     fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
#     axes[0].imshow(input_vol[z_slice, :, :], cmap='gray')
#     axes[0].set_title(f"Raw Zarr Image (Z={z_slice})")
#     axes[0].axis('off')
    
#     axes[1].imshow(target_vol[z_slice, :, :], cmap='hot')
#     axes[1].set_title("Ground Truth Mask")
#     axes[1].axis('off')
    
#     axes[2].imshow(pred_vol[z_slice, :, :], cmap='hot')
#     axes[2].set_title(f"AI Prediction (Epoch {epochs})")
#     axes[2].axis('off')
    
#     plt.tight_layout()
#     plt.show()

In [10]:
# # Auto-install dependency 
# !pip install zarr

# import os
# import numpy as np
# import zarr
# import scipy.ndimage as ndimage
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import matplotlib.pyplot as plt
# from torch.utils.data import Dataset, DataLoader

# # 1. Dataset Pipeline (Unchanged)
# class ZebrafishDataset(Dataset):
#     def __init__(self, zarr_path, geff_path, time_frame=0, mask_sigma=2.0):
#         self.root_img = zarr.open(zarr_path, mode='r')
#         self.image_tensor = self.root_img['0']
#         self.time_frame = time_frame
#         self.mask_sigma = mask_sigma
        
#         self.root_geff = zarr.open(geff_path, mode='r')
#         t_vals = self.root_geff['nodes/props/t/values'][:]
#         z_vals = self.root_geff['nodes/props/z/values'][:]
#         y_vals = self.root_geff['nodes/props/y/values'][:]
#         x_vals = self.root_geff['nodes/props/x/values'][:]
        
#         time_mask = (t_vals == self.time_frame)
#         self.target_z = z_vals[time_mask]
#         self.target_y = y_vals[time_mask]
#         self.target_x = x_vals[time_mask]

#     def __len__(self):
#         return 1 

#     def _normalize_intensity(self, volume):
#         volume = volume.astype(np.float32)
#         min_val = np.min(volume)
#         max_val = np.max(volume)
#         if max_val - min_val > 0:
#             volume = (volume - min_val) / (max_val - min_val)
#         return volume

#     def _generate_gaussian_mask(self, shape):
#         mask = np.zeros(shape, dtype=np.float32)
#         for z, y, x in zip(self.target_z, self.target_y, self.target_x):
#             z, y, x = int(z), int(y), int(x)
#             if 0 <= z < shape[0] and 0 <= y < shape[1] and 0 <= x < shape[2]:
#                 mask[z, y, x] = 1.0
                
#         mask = ndimage.gaussian_filter(mask, sigma=self.mask_sigma)
#         max_prob = np.max(mask)
#         if max_prob > 0:
#             mask = mask / max_prob
#         return mask

#     def __getitem__(self, idx):
#         raw_volume = self.image_tensor[self.time_frame, :, :, :]
#         x_normalized = self._normalize_intensity(raw_volume)
#         y_target = self._generate_gaussian_mask(x_normalized.shape)
        
#         x_tensor = torch.tensor(x_normalized).unsqueeze(0)
#         y_tensor = torch.tensor(y_target).unsqueeze(0)
#         return x_tensor, y_tensor

# # 2. UPGRADED 3D U-Net Architecture (No more Checkerboard)
# class DoubleConv3D(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.double_conv = nn.Sequential(
#             nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
#             nn.BatchNorm3d(out_channels),
#             nn.ReLU(inplace=True),
#             nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
#             nn.BatchNorm3d(out_channels),
#             nn.ReLU(inplace=True)
#         )
#     def forward(self, x):
#         return self.double_conv(x)

# class UNet3D(nn.Module):
#     def __init__(self, in_channels=1, out_channels=1, base_features=16):
#         super().__init__()
#         self.enc1 = DoubleConv3D(in_channels, base_features)
#         self.pool1 = nn.MaxPool3d(kernel_size=2, stride=2)
#         self.enc2 = DoubleConv3D(base_features, base_features * 2)
#         self.pool2 = nn.MaxPool3d(kernel_size=2, stride=2)
#         self.enc3 = DoubleConv3D(base_features * 2, base_features * 4)
#         self.pool3 = nn.MaxPool3d(kernel_size=2, stride=2)
        
#         self.bottleneck = DoubleConv3D(base_features * 4, base_features * 8)
        
#         # UPGRADE: Replaced ConvTranspose3d with smooth Upsample + Conv3d to fix checkerboarding
#         self.upconv3 = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
#             nn.Conv3d(base_features * 8, base_features * 4, kernel_size=1)
#         )
#         self.dec3 = DoubleConv3D(base_features * 8, base_features * 4)
        
#         self.upconv2 = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
#             nn.Conv3d(base_features * 4, base_features * 2, kernel_size=1)
#         )
#         self.dec2 = DoubleConv3D(base_features * 4, base_features * 2)
        
#         self.upconv1 = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
#             nn.Conv3d(base_features * 2, base_features, kernel_size=1)
#         )
#         self.dec1 = DoubleConv3D(base_features * 2, base_features)
        
#         self.out_conv = nn.Conv3d(base_features, out_channels, kernel_size=1)

#     def forward(self, x):
#         e1 = self.enc1(x)
#         p1 = self.pool1(e1)
#         e2 = self.enc2(p1)
#         p2 = self.pool2(e2)
#         e3 = self.enc3(p2)
#         p3 = self.pool3(e3)
        
#         b = self.bottleneck(p3)
        
#         d3 = self.upconv3(b)
#         d3 = torch.cat((d3, e3), dim=1)
#         d3 = self.dec3(d3)
#         d2 = self.upconv2(d3)
#         d2 = torch.cat((d2, e2), dim=1)
#         d2 = self.dec2(d2)
#         d1 = self.upconv1(d2)
#         d1 = torch.cat((d1, e1), dim=1)
#         d1 = self.dec1(d1)
        
#         out = self.out_conv(d1)
#         return torch.sigmoid(out)

# # 3. Master Execution Module
# if __name__ == "__main__":
#     print("Initializing Master Pipeline with Upgraded Architecture...")
    
#     sample_id = "6bba_cf35214c"
#     base_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
#     zarr_file = os.path.join(base_dir, f"{sample_id}.zarr")
#     geff_file = os.path.join(base_dir, f"{sample_id}.geff")
    
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"Hardware Detected: {device}")
    
#     dataset = ZebrafishDataset(zarr_path=zarr_file, geff_path=geff_file, time_frame=0)
#     dataloader = DataLoader(dataset, batch_size=1, shuffle=False)
    
#     model = UNet3D(in_channels=1, out_channels=1, base_features=16).to(device)
    
#     # UPGRADE: Using Binary Cross Entropy (BCE) instead of MSE
#     criterion = nn.BCELoss() 
#     optimizer = optim.Adam(model.parameters(), lr=1e-4)
    
#     # Increased epochs slightly to let BCE loss stabilize
#     epochs = 15 
    
#     print("\nStarting Training Loop...")
#     for epoch in range(epochs):
#         model.train()
#         epoch_loss = 0.0
        
#         for batch_idx, (inputs, targets) in enumerate(dataloader):
#             inputs = inputs.to(device)
#             targets = targets.to(device)
            
#             optimizer.zero_grad()
#             predictions = model(inputs)
#             loss = criterion(predictions, targets)
#             loss.backward()
#             optimizer.step()
            
#             epoch_loss += loss.item()
            
#         avg_loss = epoch_loss / len(dataloader)
#         print(f"Epoch [{epoch+1}/{epochs}] | BCE Loss: {avg_loss:.6f}")
        
#     print("\nTraining Complete. Executing Visualization...")
    
#     model.eval()
#     with torch.no_grad():
#         for inputs, targets in dataloader:
#             inputs = inputs.to(device)
#             predictions = model(inputs)
            
#             input_vol = inputs.cpu().squeeze().numpy()
#             target_vol = targets.cpu().squeeze().numpy()
#             pred_vol = predictions.cpu().squeeze().numpy()
#             break
            
#     # Plotting results
#     z_slice = 32 
#     fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
#     axes[0].imshow(input_vol[z_slice, :, :], cmap='gray')
#     axes[0].set_title(f"Raw Zarr Image (Z={z_slice})")
#     axes[0].axis('off')
    
#     axes[1].imshow(target_vol[z_slice, :, :], cmap='hot')
#     axes[1].set_title("Ground Truth Mask")
#     axes[1].axis('off')
    
#     axes[2].imshow(pred_vol[z_slice, :, :], cmap='hot')
#     axes[2].set_title(f"AI Prediction (Epoch {epochs})")
#     axes[2].axis('off')
    
#     plt.tight_layout()
#     plt.show()

In [11]:
# import os
# import glob
# import numpy as np
# import zarr
# import scipy.ndimage as ndimage
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader

# # 1. UPGRADED: Dynamic Multi-File Dataset Pipeline
# class FullZebrafishDataset(Dataset):
#     def __init__(self, train_dir, time_frame=0, mask_sigma=2.0):
#         self.time_frame = time_frame
#         self.mask_sigma = mask_sigma
        
#         # Scan for all .zarr files in the training directory
#         zarr_files = sorted(glob.glob(os.path.join(train_dir, "*.zarr")))
        
#         # Pair them with their corresponding .geff directories
#         self.file_pairs = []
#         for z_path in zarr_files:
#             # Replace .zarr extension with .geff
#             g_path = z_path.replace(".zarr", ".geff")
#             if os.path.exists(g_path):
#                 self.file_pairs.append((z_path, g_path))
                
#         print(f"Dataset Initialized: Found {len(self.file_pairs)} valid Zarr/GEFF pairs.")

#     def __len__(self):
#         # The dataloader will now run through ALL available files
#         return len(self.file_pairs)

#     def _normalize_intensity(self, volume):
#         volume = volume.astype(np.float32)
#         min_val = np.min(volume)
#         max_val = np.max(volume)
#         if max_val - min_val > 0:
#             volume = (volume - min_val) / (max_val - min_val)
#         return volume

#     def _generate_gaussian_mask(self, shape, target_z, target_y, target_x):
#         mask = np.zeros(shape, dtype=np.float32)
#         for z, y, x in zip(target_z, target_y, target_x):
#             z, y, x = int(z), int(y), int(x)
#             if 0 <= z < shape[0] and 0 <= y < shape[1] and 0 <= x < shape[2]:
#                 mask[z, y, x] = 1.0
                
#         mask = ndimage.gaussian_filter(mask, sigma=self.mask_sigma)
#         max_prob = np.max(mask)
#         if max_prob > 0:
#             mask = mask / max_prob
#         return mask

#     def __getitem__(self, idx):
#         # Dynamically load the specific file for this batch iteration
#         zarr_path, geff_path = self.file_pairs[idx]
        
#         # Open Zarr image
#         root_img = zarr.open(zarr_path, mode='r')
#         image_tensor = root_img['0']
        
#         # Open GEFF graph and extract coordinates
#         root_geff = zarr.open(geff_path, mode='r')
#         t_vals = root_geff['nodes/props/t/values'][:]
#         z_vals = root_geff['nodes/props/z/values'][:]
#         y_vals = root_geff['nodes/props/y/values'][:]
#         x_vals = root_geff['nodes/props/x/values'][:]
        
#         time_mask = (t_vals == self.time_frame)
#         target_z = z_vals[time_mask]
#         target_y = y_vals[time_mask]
#         target_x = x_vals[time_mask]
        
#         # Extract volume and generate mask
#         raw_volume = image_tensor[self.time_frame, :, :, :]
#         x_normalized = self._normalize_intensity(raw_volume)
#         y_target = self._generate_gaussian_mask(x_normalized.shape, target_z, target_y, target_x)
        
#         x_tensor = torch.tensor(x_normalized).unsqueeze(0)
#         y_tensor = torch.tensor(y_target).unsqueeze(0)
        
#         return x_tensor, y_tensor


# # 2. 3D U-Net Architecture (Remains the same - it works perfectly)
# class DoubleConv3D(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.double_conv = nn.Sequential(
#             nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
#             nn.BatchNorm3d(out_channels),
#             nn.ReLU(inplace=True),
#             nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
#             nn.BatchNorm3d(out_channels),
#             nn.ReLU(inplace=True)
#         )
#     def forward(self, x):
#         return self.double_conv(x)

# class UNet3D(nn.Module):
#     def __init__(self, in_channels=1, out_channels=1, base_features=16):
#         super().__init__()
#         self.enc1 = DoubleConv3D(in_channels, base_features)
#         self.pool1 = nn.MaxPool3d(kernel_size=2, stride=2)
#         self.enc2 = DoubleConv3D(base_features, base_features * 2)
#         self.pool2 = nn.MaxPool3d(kernel_size=2, stride=2)
#         self.enc3 = DoubleConv3D(base_features * 2, base_features * 4)
#         self.pool3 = nn.MaxPool3d(kernel_size=2, stride=2)
        
#         self.bottleneck = DoubleConv3D(base_features * 4, base_features * 8)
        
#         self.upconv3 = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
#             nn.Conv3d(base_features * 8, base_features * 4, kernel_size=1)
#         )
#         self.dec3 = DoubleConv3D(base_features * 8, base_features * 4)
        
#         self.upconv2 = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
#             nn.Conv3d(base_features * 4, base_features * 2, kernel_size=1)
#         )
#         self.dec2 = DoubleConv3D(base_features * 4, base_features * 2)
        
#         self.upconv1 = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
#             nn.Conv3d(base_features * 2, base_features, kernel_size=1)
#         )
#         self.dec1 = DoubleConv3D(base_features * 2, base_features)
        
#         self.out_conv = nn.Conv3d(base_features, out_channels, kernel_size=1)

#     def forward(self, x):
#         e1 = self.enc1(x)
#         p1 = self.pool1(e1)
#         e2 = self.enc2(p1)
#         p2 = self.pool2(e2)
#         e3 = self.enc3(p2)
#         p3 = self.pool3(e3)
        
#         b = self.bottleneck(p3)
        
#         d3 = self.upconv3(b)
#         d3 = torch.cat((d3, e3), dim=1)
#         d3 = self.dec3(d3)
#         d2 = self.upconv2(d3)
#         d2 = torch.cat((d2, e2), dim=1)
#         d2 = self.dec2(d2)
#         d1 = self.upconv1(d2)
#         d1 = torch.cat((d1, e1), dim=1)
#         d1 = self.dec1(d1)
        
#         out = self.out_conv(d1)
#         return torch.sigmoid(out)


# # 3. Scaled Execution Module
# if __name__ == "__main__":
#     print("Initializing Full Dataset Pipeline...")
    
#     train_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"Hardware Detected: {device}")
    
#     # Initialize the upgraded dataset (This will load ALL files)
#     full_dataset = FullZebrafishDataset(train_dir=train_dir, time_frame=0)
    
#     # Optional: For testing purposes, we use a subset so you don't wait 3 hours for an epoch.
#     # We will slice the dataset to use only the first 10 files for this trial run.
#     subset_dataset = torch.utils.data.Subset(full_dataset, range(10)) 
#     dataloader = DataLoader(subset_dataset, batch_size=1, shuffle=True)
    
#     model = UNet3D(in_channels=1, out_channels=1, base_features=16).to(device)
#     criterion = nn.BCELoss() 
#     optimizer = optim.Adam(model.parameters(), lr=1e-4)
    
#     epochs = 5 
    
#     print("\nStarting Training Loop on Dataset Subset...")
#     for epoch in range(epochs):
#         model.train()
#         epoch_loss = 0.0
        
#         for batch_idx, (inputs, targets) in enumerate(dataloader):
#             inputs = inputs.to(device)
#             targets = targets.to(device)
            
#             optimizer.zero_grad()
#             predictions = model(inputs)
#             loss = criterion(predictions, targets)
#             loss.backward()
#             optimizer.step()
            
#             epoch_loss += loss.item()
#             print(f"  -> Processed file {batch_idx+1}/{len(dataloader)} - Current Loss: {loss.item():.4f}")
            
#         avg_loss = epoch_loss / len(dataloader)
#         print(f"=== Epoch [{epoch+1}/{epochs}] Completed | Average BCE Loss: {avg_loss:.6f} ===\n")

In [12]:
# import os
# import glob
# import numpy as np
# import zarr
# import scipy.ndimage as ndimage
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader

# # 1. UPGRADED: Dynamic Multi-File Dataset Pipeline
# class FullZebrafishDataset(Dataset):
#     def __init__(self, train_dir, time_frame=0, mask_sigma=2.0):
#         self.time_frame = time_frame
#         self.mask_sigma = mask_sigma
        
#         # Scan for all .zarr files in the training directory
#         zarr_files = sorted(glob.glob(os.path.join(train_dir, "*.zarr")))
        
#         # Pair them with their corresponding .geff directories
#         self.file_pairs = []
#         for z_path in zarr_files:
#             # Replace .zarr extension with .geff
#             g_path = z_path.replace(".zarr", ".geff")
#             if os.path.exists(g_path):
#                 self.file_pairs.append((z_path, g_path))
                
#         print(f"Dataset Initialized: Found {len(self.file_pairs)} valid Zarr/GEFF pairs.")

#     def __len__(self):
#         # The dataloader will now run through ALL available files
#         return len(self.file_pairs)

#     def _normalize_intensity(self, volume):
#         volume = volume.astype(np.float32)
#         min_val = np.min(volume)
#         max_val = np.max(volume)
#         if max_val - min_val > 0:
#             volume = (volume - min_val) / (max_val - min_val)
#         return volume

#     def _generate_gaussian_mask(self, shape, target_z, target_y, target_x):
#         mask = np.zeros(shape, dtype=np.float32)
#         for z, y, x in zip(target_z, target_y, target_x):
#             z, y, x = int(z), int(y), int(x)
#             if 0 <= z < shape[0] and 0 <= y < shape[1] and 0 <= x < shape[2]:
#                 mask[z, y, x] = 1.0
                
#         mask = ndimage.gaussian_filter(mask, sigma=self.mask_sigma)
#         max_prob = np.max(mask)
#         if max_prob > 0:
#             mask = mask / max_prob
#         return mask

#     def __getitem__(self, idx):
#         # Dynamically load the specific file for this batch iteration
#         zarr_path, geff_path = self.file_pairs[idx]
        
#         # Open Zarr image
#         root_img = zarr.open(zarr_path, mode='r')
#         image_tensor = root_img['0']
        
#         # Open GEFF graph and extract coordinates
#         root_geff = zarr.open(geff_path, mode='r')
#         t_vals = root_geff['nodes/props/t/values'][:]
#         z_vals = root_geff['nodes/props/z/values'][:]
#         y_vals = root_geff['nodes/props/y/values'][:]
#         x_vals = root_geff['nodes/props/x/values'][:]
        
#         time_mask = (t_vals == self.time_frame)
#         target_z = z_vals[time_mask]
#         target_y = y_vals[time_mask]
#         target_x = x_vals[time_mask]
        
#         # Extract volume and generate mask
#         raw_volume = image_tensor[self.time_frame, :, :, :]
#         x_normalized = self._normalize_intensity(raw_volume)
#         y_target = self._generate_gaussian_mask(x_normalized.shape, target_z, target_y, target_x)
        
#         x_tensor = torch.tensor(x_normalized).unsqueeze(0)
#         y_tensor = torch.tensor(y_target).unsqueeze(0)
        
#         return x_tensor, y_tensor


# # 2. 3D U-Net Architecture (Remains the same - it works perfectly)
# class DoubleConv3D(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.double_conv = nn.Sequential(
#             nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
#             nn.BatchNorm3d(out_channels),
#             nn.ReLU(inplace=True),
#             nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
#             nn.BatchNorm3d(out_channels),
#             nn.ReLU(inplace=True)
#         )
#     def forward(self, x):
#         return self.double_conv(x)

# class UNet3D(nn.Module):
#     def __init__(self, in_channels=1, out_channels=1, base_features=16):
#         super().__init__()
#         self.enc1 = DoubleConv3D(in_channels, base_features)
#         self.pool1 = nn.MaxPool3d(kernel_size=2, stride=2)
#         self.enc2 = DoubleConv3D(base_features, base_features * 2)
#         self.pool2 = nn.MaxPool3d(kernel_size=2, stride=2)
#         self.enc3 = DoubleConv3D(base_features * 2, base_features * 4)
#         self.pool3 = nn.MaxPool3d(kernel_size=2, stride=2)
        
#         self.bottleneck = DoubleConv3D(base_features * 4, base_features * 8)
        
#         self.upconv3 = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
#             nn.Conv3d(base_features * 8, base_features * 4, kernel_size=1)
#         )
#         self.dec3 = DoubleConv3D(base_features * 8, base_features * 4)
        
#         self.upconv2 = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
#             nn.Conv3d(base_features * 4, base_features * 2, kernel_size=1)
#         )
#         self.dec2 = DoubleConv3D(base_features * 4, base_features * 2)
        
#         self.upconv1 = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
#             nn.Conv3d(base_features * 2, base_features, kernel_size=1)
#         )
#         self.dec1 = DoubleConv3D(base_features * 2, base_features)
        
#         self.out_conv = nn.Conv3d(base_features, out_channels, kernel_size=1)

#     def forward(self, x):
#         e1 = self.enc1(x)
#         p1 = self.pool1(e1)
#         e2 = self.enc2(p1)
#         p2 = self.pool2(e2)
#         e3 = self.enc3(p2)
#         p3 = self.pool3(e3)
        
#         b = self.bottleneck(p3)
        
#         d3 = self.upconv3(b)
#         d3 = torch.cat((d3, e3), dim=1)
#         d3 = self.dec3(d3)
#         d2 = self.upconv2(d3)
#         d2 = torch.cat((d2, e2), dim=1)
#         d2 = self.dec2(d2)
#         d1 = self.upconv1(d2)
#         d1 = torch.cat((d1, e1), dim=1)
#         d1 = self.dec1(d1)
        
#         out = self.out_conv(d1)
#         return torch.sigmoid(out)


# # 3. Scaled Execution Module
# if __name__ == "__main__":
#     print("Initializing Full Dataset Pipeline...")
    
#     train_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"Hardware Detected: {device}")
    
#     # Initialize the upgraded dataset (This will load ALL files)
#     full_dataset = FullZebrafishDataset(train_dir=train_dir, time_frame=0)
    
#     # Optional: For testing purposes, we use a subset so you don't wait 3 hours for an epoch.
#     # We will slice the dataset to use only the first 10 files for this trial run.
#     subset_dataset = torch.utils.data.Subset(full_dataset, range(10)) 
#     dataloader = DataLoader(subset_dataset, batch_size=1, shuffle=True)
    
#     model = UNet3D(in_channels=1, out_channels=1, base_features=16).to(device)
#     criterion = nn.BCELoss() 
#     optimizer = optim.Adam(model.parameters(), lr=1e-4)
    
#     epochs = 5 
    
#     print("\nStarting Training Loop on Dataset Subset...")
#     for epoch in range(epochs):
#         model.train()
#         epoch_loss = 0.0
        
#         for batch_idx, (inputs, targets) in enumerate(dataloader):
#             inputs = inputs.to(device)
#             targets = targets.to(device)
            
#             optimizer.zero_grad()
#             predictions = model(inputs)
#             loss = criterion(predictions, targets)
#             loss.backward()
#             optimizer.step()
            
#             epoch_loss += loss.item()
#             print(f"  -> Processed file {batch_idx+1}/{len(dataloader)} - Current Loss: {loss.item():.4f}")
            
#         avg_loss = epoch_loss / len(dataloader)
#         print(f"=== Epoch [{epoch+1}/{epochs}] Completed | Average BCE Loss: {avg_loss:.6f} ===\n")

In [13]:
# # 3. Production Execution Module

# if __name__ == "__main__":
#     print("Initializing Full Dataset Pipeline...")
    
#     # Environment Configuration
#     train_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"Hardware Detected: {device}")
    
#     # Dataset and DataLoader Initialization
#     full_dataset = FullZebrafishDataset(train_dir=train_dir, time_frame=0)
#     dataloader = DataLoader(full_dataset, batch_size=1, shuffle=True)
    
#     # Model, Loss Function, and Optimizer Initialization
#     model = UNet3D(in_channels=1, out_channels=1, base_features=16).to(device)
#     criterion = nn.BCELoss() 
#     optimizer = optim.Adam(model.parameters(), lr=1e-4)
    
#     # Hyperparameter Configuration
#     epochs = 10 
    
#     print(f"\nStarting Full-Scale Training Loop on {len(full_dataset)} samples...")
    
#     for epoch in range(epochs):
#         model.train()
#         epoch_loss = 0.0
        
#         for batch_idx, (inputs, targets) in enumerate(dataloader):
#             # Transfer tensors to the configured device (GPU/CPU)
#             inputs = inputs.to(device)
#             targets = targets.to(device)
            
#             # Reset gradients for the current batch
#             optimizer.zero_grad()
            
#             # Forward pass
#             predictions = model(inputs)
            
#             # Compute loss and perform backpropagation
#             loss = criterion(predictions, targets)
#             loss.backward()
#             optimizer.step()
            
#             # Accumulate loss for epoch averaging
#             epoch_loss += loss.item()
            
#             # Log progress every 20 batches to maintain a clean terminal
#             if (batch_idx + 1) % 20 == 0:
#                 print(f"  -> Processed file {batch_idx+1:03d}/{len(dataloader)} - Current Loss: {loss.item():.4f}")
            
#         # Calculate and display the average loss for the epoch
#         avg_loss = epoch_loss / len(dataloader)
#         print(f"=== Epoch [{epoch+1}/{epochs}] Completed | Average BCE Loss: {avg_loss:.6f} ===")
        
#         # Model Checkpointing: Save weights after each epoch
#         save_path = f"unet3d_zebrafish_epoch_{epoch+1}.pth"
#         torch.save(model.state_dict(), save_path)
#         print(f"[*] Model weights saved successfully to: {save_path}\n")
        
#     print("Full-scale training completed successfully.")

In [14]:
# !pip install zarr pandas

# import os
# import glob
# import numpy as np
# import pandas as pd
# import zarr
# import torch
# import torch.nn as nn
# from scipy.ndimage import label, center_of_mass

# # 1. 3D U-Net Architecture (Required to load the saved weights)
# class DoubleConv3D(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.double_conv = nn.Sequential(
#             nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
#             nn.BatchNorm3d(out_channels),
#             nn.ReLU(inplace=True),
#             nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
#             nn.BatchNorm3d(out_channels),
#             nn.ReLU(inplace=True)
#         )
#     def forward(self, x):
#         return self.double_conv(x)

# class UNet3D(nn.Module):
#     def __init__(self, in_channels=1, out_channels=1, base_features=16):
#         super().__init__()
#         self.enc1 = DoubleConv3D(in_channels, base_features)
#         self.pool1 = nn.MaxPool3d(kernel_size=2, stride=2)
#         self.enc2 = DoubleConv3D(base_features, base_features * 2)
#         self.pool2 = nn.MaxPool3d(kernel_size=2, stride=2)
#         self.enc3 = DoubleConv3D(base_features * 2, base_features * 4)
#         self.pool3 = nn.MaxPool3d(kernel_size=2, stride=2)
        
#         self.bottleneck = DoubleConv3D(base_features * 4, base_features * 8)
        
#         self.upconv3 = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
#             nn.Conv3d(base_features * 8, base_features * 4, kernel_size=1)
#         )
#         self.dec3 = DoubleConv3D(base_features * 8, base_features * 4)
        
#         self.upconv2 = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
#             nn.Conv3d(base_features * 4, base_features * 2, kernel_size=1)
#         )
#         self.dec2 = DoubleConv3D(base_features * 4, base_features * 2)
        
#         self.upconv1 = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
#             nn.Conv3d(base_features * 2, base_features, kernel_size=1)
#         )
#         self.dec1 = DoubleConv3D(base_features * 2, base_features)
        
#         self.out_conv = nn.Conv3d(base_features, out_channels, kernel_size=1)

#     def forward(self, x):
#         e1 = self.enc1(x)
#         p1 = self.pool1(e1)
#         e2 = self.enc2(p1)
#         p2 = self.pool2(e2)
#         e3 = self.enc3(p2)
#         p3 = self.pool3(e3)
#         b = self.bottleneck(p3)
#         d3 = self.upconv3(b)
#         d3 = torch.cat((d3, e3), dim=1)
#         d3 = self.dec3(d3)
#         d2 = self.upconv2(d3)
#         d2 = torch.cat((d2, e2), dim=1)
#         d2 = self.dec2(d2)
#         d1 = self.upconv1(d2)
#         d1 = torch.cat((d1, e1), dim=1)
#         d1 = self.dec1(d1)
#         out = self.out_conv(d1)
#         return torch.sigmoid(out)

# # 2. Post-Processing Module
# def extract_coordinates(probability_mask, threshold=0.5):
#     binary_mask = probability_mask > threshold
#     labeled_mask, num_features = label(binary_mask)
#     centroids = center_of_mass(binary_mask, labeled_mask, range(1, num_features + 1))
#     return centroids

# # 3. Kaggle Submission Generator (PURE INFERENCE, NO TRAINING)
# if __name__ == "__main__":
#     print("Initiating Kaggle Submission Pipeline...")
    
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"Hardware Detected: {device}")
    
#     # LOAD THE AI BRAIN FROM YOUR KAGGLE DATASET
#     # If you have epoch_10 in your dataset, change the number '1' below to '10'.
#     weights_path = "/kaggle/input/datasets/gedebhayuadhipramana/zebrafish-3d-u-net-weights/unet3d_zebrafish_epoch_10.pth"
    
#     model = UNet3D(in_channels=1, out_channels=1, base_features=16).to(device)
    
#     if os.path.exists(weights_path):
#         # We are only 'reading' (load_state_dict), absolutely no 'saving' (torch.save)
#         model.load_state_dict(torch.load(weights_path, map_location=device))
#         model.eval()
#         print(f"Production weights loaded from: {weights_path}")
#     else:
#         print(f"Error: Weights not found at {weights_path}. Please verify the file name.")
        
#     # Scan the TEST folder to predict unseen data
#     test_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/test"
    
#     if not os.path.exists(test_dir):
#         print(f"Error: Test directory not found at {test_dir}.")
#     else:
#         test_zarr_files = sorted(glob.glob(os.path.join(test_dir, "*.zarr")))
#         print(f"Found {len(test_zarr_files)} test samples to process.")
        
#         submission_data = []
        
#         # Begin Prediction Loop
#         with torch.no_grad():
#             for zarr_path in test_zarr_files:
#                 sample_id = os.path.basename(zarr_path).replace('.zarr', '')
#                 root_img = zarr.open(zarr_path, mode='r')
                
#                 time_frame = 0 
#                 raw_vol = root_img['0'][time_frame, :, :, :].astype(np.float32)
                
#                 min_val, max_val = np.min(raw_vol), np.max(raw_vol)
#                 if max_val - min_val > 0:
#                     raw_vol = (raw_vol - min_val) / (max_val - min_val)
                    
#                 input_tensor = torch.tensor(raw_vol).unsqueeze(0).unsqueeze(0).to(device)
                
#                 # The AI predicts the cell locations
#                 prediction_tensor = model(input_tensor)
#                 prediction_mask = prediction_tensor.cpu().squeeze().numpy()
                
#                 # Extract coordinates into numerical values
#                 extracted_coords = extract_coordinates(prediction_mask, threshold=0.5)
                
#                 # Format to Kaggle's official submission standard
#                 for z, y, x in extracted_coords:
#                     submission_data.append({
#                         "id": sample_id,
#                         "t": time_frame,
#                         "z": z,
#                         "y": y,
#                         "x": x
#                     })
                    
#         # Generate the final submission.csv file
#         if submission_data:
#             submission_df = pd.DataFrame(submission_data)
#             output_csv = "/kaggle/working/submission.csv" # We are allowed to save CSVs in /kaggle/working/
#             submission_df.to_csv(output_csv, index=False)
#             print(f"\nSuccess! Submission file generated: {output_csv}")
#             print(submission_df.head())
#         else:
#             print("\nNo cells were detected. Try lowering the threshold in extract_coordinates.")

In [15]:
# !pip install zarr pandas

# import os
# import glob
# import numpy as np
# import pandas as pd
# import zarr
# import torch
# import torch.nn as nn
# from scipy.ndimage import label, center_of_mass

# # 1. 3D U-Net Architecture (Required to load the saved weights)
# class DoubleConv3D(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.double_conv = nn.Sequential(
#             nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
#             nn.BatchNorm3d(out_channels),
#             nn.ReLU(inplace=True),
#             nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
#             nn.BatchNorm3d(out_channels),
#             nn.ReLU(inplace=True)
#         )
#     def forward(self, x):
#         return self.double_conv(x)

# class UNet3D(nn.Module):
#     def __init__(self, in_channels=1, out_channels=1, base_features=16):
#         super().__init__()
#         self.enc1 = DoubleConv3D(in_channels, base_features)
#         self.pool1 = nn.MaxPool3d(kernel_size=2, stride=2)
#         self.enc2 = DoubleConv3D(base_features, base_features * 2)
#         self.pool2 = nn.MaxPool3d(kernel_size=2, stride=2)
#         self.enc3 = DoubleConv3D(base_features * 2, base_features * 4)
#         self.pool3 = nn.MaxPool3d(kernel_size=2, stride=2)
        
#         self.bottleneck = DoubleConv3D(base_features * 4, base_features * 8)
        
#         self.upconv3 = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
#             nn.Conv3d(base_features * 8, base_features * 4, kernel_size=1)
#         )
#         self.dec3 = DoubleConv3D(base_features * 8, base_features * 4)
        
#         self.upconv2 = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
#             nn.Conv3d(base_features * 4, base_features * 2, kernel_size=1)
#         )
#         self.dec2 = DoubleConv3D(base_features * 4, base_features * 2)
        
#         self.upconv1 = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True),
#             nn.Conv3d(base_features * 2, base_features, kernel_size=1)
#         )
#         self.dec1 = DoubleConv3D(base_features * 2, base_features)
        
#         self.out_conv = nn.Conv3d(base_features, out_channels, kernel_size=1)

#     def forward(self, x):
#         e1 = self.enc1(x)
#         p1 = self.pool1(e1)
#         e2 = self.enc2(p1)
#         p2 = self.pool2(e2)
#         e3 = self.enc3(p2)
#         p3 = self.pool3(e3)
#         b = self.bottleneck(p3)
#         d3 = self.upconv3(b)
#         d3 = torch.cat((d3, e3), dim=1)
#         d3 = self.dec3(d3)
#         d2 = self.upconv2(d3)
#         d2 = torch.cat((d2, e2), dim=1)
#         d2 = self.dec2(d2)
#         d1 = self.upconv1(d2)
#         d1 = torch.cat((d1, e1), dim=1)
#         d1 = self.dec1(d1)
#         out = self.out_conv(d1)
#         return torch.sigmoid(out)

# # 2. Post-Processing Module
# def extract_coordinates(probability_mask, threshold=0.5):
#     binary_mask = probability_mask > threshold
#     labeled_mask, num_features = label(binary_mask)
#     centroids = center_of_mass(binary_mask, labeled_mask, range(1, num_features + 1))
#     return centroids

# # 3. Kaggle Submission Generator (PURE INFERENCE, NO TRAINING)
# if __name__ == "__main__":
#     print("Initiating Kaggle Submission Pipeline...")
    
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"Hardware Detected: {device}")
    
#     # LOAD THE AI BRAIN FROM YOUR KAGGLE DATASET
#     # If you have epoch_10 in your dataset, change the number '1' below to '10'.
#     weights_path = "/kaggle/input/zebrafish-3d-u-net-weights/unet3d_zebrafish_epoch_1.pth" 
    
#     model = UNet3D(in_channels=1, out_channels=1, base_features=16).to(device)
    
#     if os.path.exists(weights_path):
#         # We are only 'reading' (load_state_dict), absolutely no 'saving' (torch.save)
#         model.load_state_dict(torch.load(weights_path, map_location=device))
#         model.eval()
#         print(f"Production weights loaded from: {weights_path}")
#     else:
#         print(f"Error: Weights not found at {weights_path}. Please verify the file name.")
        
#     # Scan the TEST folder to predict unseen data
#     test_dir = "/kaggle/input/competitions/biohub-cell-tracking-during-development/test"
    
#     if not os.path.exists(test_dir):
#         print(f"Error: Test directory not found at {test_dir}.")
#     else:
#         test_zarr_files = sorted(glob.glob(os.path.join(test_dir, "*.zarr")))
#         print(f"Found {len(test_zarr_files)} test samples to process.")
        
#         submission_data = []
        
#         # Begin Prediction Loop
#         with torch.no_grad():
#             for zarr_path in test_zarr_files:
#                 sample_id = os.path.basename(zarr_path).replace('.zarr', '')
#                 root_img = zarr.open(zarr_path, mode='r')
                
#                 time_frame = 0 
#                 raw_vol = root_img['0'][time_frame, :, :, :].astype(np.float32)
                
#                 min_val, max_val = np.min(raw_vol), np.max(raw_vol)
#                 if max_val - min_val > 0:
#                     raw_vol = (raw_vol - min_val) / (max_val - min_val)
                    
#                 input_tensor = torch.tensor(raw_vol).unsqueeze(0).unsqueeze(0).to(device)
                
#                 # The AI predicts the cell locations
#                 prediction_tensor = model(input_tensor)
#                 prediction_mask = prediction_tensor.cpu().squeeze().numpy()
                
#                 # Extract coordinates into numerical values
#                 extracted_coords = extract_coordinates(prediction_mask, threshold=0.5)
                
#                 # Format to Kaggle's official submission standard
#                 for z, y, x in extracted_coords:
#                     submission_data.append({
#                         "id": sample_id,
#                         "t": time_frame,
#                         "z": z,
#                         "y": y,
#                         "x": x
#                     })
                    
#         # Generate the final submission.csv file
#         if submission_data:
#             submission_df = pd.DataFrame(submission_data)
#             output_csv = "/kaggle/working/submission.csv" # We are allowed to save CSVs in /kaggle/working/
#             submission_df.to_csv(output_csv, index=False)
#             print(f"\nSuccess! Submission file generated: {output_csv}")
#             print(submission_df.head())
#         else:
#             print("\nNo cells were detected. Try lowering the threshold in extract_coordinates.")